# 노트북 06 — 실제 시스템과의 비교

노트북 05는 우리가 *무엇을* 생략했는지 나열했습니다. 이 챕터는 pq-messenger를 실제로 배포된 프로토콜의 지도 위에 놓아 봅니다: Signal Double Ratchet, Apple iMessage PQ3, WireGuard, 그리고 MLS. 점수를 매기는 게 목적이 아니라 — *각 시스템이 어떤 설계 문제를 풀고 있는지*, 그리고 pq-messenger가 교육 도구로서 어디에 위치하는지를 보는 것이 목적입니다.

## 4개 시스템을 한 페이지에

| 측면                     | **pq-messenger**            | **Signal (Double Ratchet)** | **iMessage PQ3**          | **WireGuard**                  | **MLS (RFC 9420)**             |
| ------------------------ | --------------------------- | --------------------------- | ------------------------- | ------------------------------ | ------------------------------ |
| 주요 용도                | 교육, 1:1 데모              | 1:1 메시징                  | 1:1 + 소규모 그룹 iMsg    | VPN 터널 (point-to-point)      | 대규모 그룹 메시징             |
| 초기 키 합의             | 하이브리드 X3DH (X25519+ML-KEM) | X3DH (X25519, 고전)     | 하이브리드 (ECDH+ML-KEM)  | Noise IK (X25519)              | 트리 기반, 그룹 단위           |
| 지속적 재키잉           | 대칭 + DH (v0.2부터)                      | Double ratchet (DH+대칭)    | DR + 주기적 PQ 재키잉     | 핸드셰이크마다, 약 2분 주기    | 트리 기반 래칫                 |
| 포스트 양자?             | 예 (KEM 절반 하이브리드)    | 아직 (계획 있음)            | 예 (iOS 17.4부터)         | 아니오 (아직)                  | 플러그형; PQ 시픈슈트 작업 중  |
| 인증                     | TOFU                        | Ed25519 + safety number     | Apple ID + Contact Verif. | 사전 공유 정적 공개키          | Credential + 트리 서명         |
| 그룹 채팅                | 없음                        | Sender Keys                 | Pairwise 세션             | N/A                            | 네이티브, 수천 명 확장         |
| 멀티 디바이스            | 없음                        | 디바이스별 세션             | iCloud Keychain 동기화    | 피어당 한 키 (수동)            | 그룹 트리의 leaf별             |
| 메타데이터 프라이버시    | 없음                        | Sealed sender               | Apple ID 최소화           | 서버리스 설계                  | 범위 외 (전송 계층 일)         |
| 비동기(오프라인) 전송    | 파일 폴링                   | 서버 큐 + 푸시              | APNs                      | 없음 (항상 켜진 터널)          | 서버 매개                      |
| 표준                     | 이 노트북                   | Signal 명세                 | Apple 내부                | RFC informational              | RFC 9420                       |

## 짝지어 본 관찰

### pq-messenger vs Signal Double Ratchet

**모양**은 같습니다: 장기 신원, 임시 핸드셰이크, 루트 키, 방향별 체인 키, 메시지마다 AEAD. v0.2부터 **격차가 메워졌습니다**: pq-messenger는 이제 완전한 Double Ratchet (대칭 + DH 절반)을 실행합니다. 남은 차이는 운영적인 부분 — 그룹 채팅, 멀티 디바이스, sealed sender — 이지 암호학적 코어가 아닙니다.

vanilla Signal이 *아직* 가지고 있지 *않은* 것을 pq-messenger는 가지고 있습니다: 초기 핸드셰이크의 포스트 양자 절반. Signal은 2023년에 우리와 같은 하이브리드 단계를 수행하는 [PQXDH 제안](https://signal.org/docs/specifications/pqxdh/)을 발표했고, 롤아웃이 진행 중입니다.

### pq-messenger vs iMessage PQ3

PQ3는 *프로덕션* 하이브리드 PQ 메신저가 어떤 모습인지를 보여줍니다. Apple의 [기술 보고서](https://security.apple.com/blog/imessage-pq3/) (2024)는 우리가 여기서 가르치는 것에 가장 가까운 실세계 유사품이며, 우리는 당기지 않는 세 가지 레버를 가지고 있습니다:

1. **주기적 PQ 재키잉** — 매 N개 메시지 또는 M분마다 새로운 ML-KEM 캡슐화를 실행해 엔트로피를 갱신. 대화 도중에도 "harvest now, decrypt later" 공격 윈도우를 닫습니다.
2. **Contact Key Verification** — identity 키 변경을 사용자에게 노출. TOFU의 치료법.
3. **AEAD에서의 하이브리드 commitment** — AEAD의 associated data가 메시지를 *고전과 PQ 비밀 모두*에 묶어, 한 절반의 미래 깨짐이 메시지를 소급해 위조하지 못하게 함.

### pq-messenger vs WireGuard

WireGuard는 메신저가 아니라 **터널**입니다. 핸드셰이크(Noise IK)가 기본적으로 *약 2분마다* 실행되므로, post-compromise security는 래칫팅이 아니라 재핸드셰이크로 달성됩니다. 암호학적으로는 더 단순합니다 — 건너뛴 키 캐시도, 래칫 상태도 없음 — 그러나 그 대가로 대역폭과 long-lived TCP/UDP 연결 가정이 따라옵니다. WireGuard는 의도적으로 포스트 양자가 *아닙니다*; 프로젝트의 입장은 "PQ KEM 표준이 성숙하기를 기다린다"입니다.

### pq-messenger vs MLS

다른 문제입니다. MLS(RFC 9420)는 수천 명의 그룹을 위해 만들어졌습니다; "래칫"이 이진 트리이며, 멤버 추가/제거 시 $O(\log N)$ 키만 갱신합니다. WhatsApp과 Discord는 정확히 이 이유로 Sender Keys에서 MLS로 옮겼습니다. 두 사람에게는 MLS가 과합니다; pq-messenger는 크기 스펙트럼의 반대편 끝에 있습니다.

## 이 지도의 쓸모

실제 프로토콜 명세 — Signal, Apple, IETF — 를 읽으면 신원 프로비저닝, 멀티 디바이스, 어뷰즈 신고, 그룹 멤버십 관리, 키 투명성에 관한 수백 페이지를 보게 됩니다. 그 중 대부분은 **암호학적 코어와는 직교**합니다. 코어는 우리가 만든 것:

- 고전과 양자 적 모두에 대해 살아남는 핸드셰이크.
- 현재가 노출되어도 과거 메시지를 보호하는 래칫.
- 모든 바이트의 변조를 탐지하는 AEAD.

이 셋이 동작하면, 나머지는 엔지니어링입니다 — 중요하고 어렵지만, 다른 종류의 어려움입니다. 노트북 05는 "나머지"가 실제로 무엇을 담고 있는지에 대한 목록입니다.

## 참고 문헌

- Signal Foundation. [*The Double Ratchet Algorithm*](https://signal.org/docs/specifications/doubleratchet/). 2016.
- Signal Foundation. [*The X3DH Key Agreement Protocol*](https://signal.org/docs/specifications/x3dh/). 2016.
- Signal Foundation. [*PQXDH — A Post-Quantum Extended Diffie-Hellman Key Agreement*](https://signal.org/docs/specifications/pqxdh/). 2023.
- Apple Security Engineering. [*iMessage with PQ3: The new state of the art in quantum-secure messaging at scale*](https://security.apple.com/blog/imessage-pq3/). 2024.
- Donenfeld, Jason. [*WireGuard: Next Generation Kernel Network Tunnel*](https://www.wireguard.com/papers/wireguard.pdf). NDSS 2017.
- IETF MLS WG. [*The Messaging Layer Security (MLS) Protocol*](https://datatracker.ietf.org/doc/html/rfc9420). RFC 9420. 2023.